In [2]:
import csv
import json
import os
from PIL import Image
from io import BytesIO
import base64
import pandas as pd
import pickle
import random
import numpy as np
from tqdm import tqdm

In [23]:
ws = '/home/guests/sebastian_sotela'
data_dir = f'{ws}/PLURAL/data'
root = '/home/polyaxon-data/data1/mimic'
mimic_cxr_jpg = f'{root}/mimic-cxr-jpg/2.0.0'
metadata_file = f'{mimic_cxr_jpg}/mimic-cxr-2.0.0-metadata.csv'
images_path = f'{mimic_cxr_jpg}/files'
diff_vqa_path = f'{root}/diff_vqa/physionet.org/files/medical-diff-vqa/1.0.0'
qa_file = f'{diff_vqa_path}/mimic_pair_questions.csv'
mimic_all_file = f'{diff_vqa_path}/mimic_all.csv'

task = 'findings'
split = 'test'

df_metadata = pd.read_csv(metadata_file)
df_qa = pd.read_csv(qa_file)
df_mimic_all = pd.read_csv(mimic_all_file)
# TODO 
# loop, pickle file 
if task == 'qa':
    dataset_name = 'mimic_qa'

    df1 = df_qa[['subject_id', 'study_id', 'ref_id', 'question', 'answer', 'split']]
    df2 = df_mimic_all[['study_id', 'dicom_id']]
    # merge to add dicom_id
    merged_df = pd.merge(df1, df2, on='study_id', how='left')
    #merge to add ref id
    merged_df = pd.merge(merged_df, df2, left_on='ref_id', right_on='study_id', how='left', suffixes=('', '_ref'))
    # Drop duplicate study_id column
    merged_df.drop(columns=['ref_id'], inplace=True)
    # FIXME
    # Consider the loop
    df_r = merged_df[merged_df['split'] == split]
    print(f'{df_r.head()}\n\nSize: {merged_df.shape[0]}')

else:
    if task == 'findings':
        question = 'what does the image describe?'
        dataset_name = 'mimic_f'
    else:
        #task == impressions
        question = 'what is the summary of the image?'
        dataset_name = 'mimic_i'
    df_r = pd.read_csv(f'{ws}/mimic_data/mimic_{split}_{task}.csv')

In [24]:
import re

# Filter empty or nonsense cells
# Add a column for the length of each report
df = df_r
df['report_length'] = df['report'].apply(len)
#patterns = [' .','as above .', 'see above .', 'as compared to  .']

#for pattern in patterns:
#    df_filtered = df[df['report'] == pattern]
    #print(pattern, df_filtered.shape[0], sep='\n')
#filtered_df = df[~df['report'].isin(patterns)]

patterns = [' .', 'is .', 'am .', 'be .', 'in .', 'at the .', 'see below .', 'send the .', 'see below .',
             ' through  .', 'is and for .', 'ct chest of  .', 'chest radiograph  .', 'no previous images .', 
             'see impression below .', 'prior chest radiographs  .', 'portable chest radiograph .', 
             'ct scan of the torso from  .','radiographs are under penetrated .','portable ap upright chest  at 1722 .',
             'prior chest radiographs  through 47 .','portable ap approach vessel  at 0739 .',
             'chest radiographs since  most recently  .','prior chest radiographs  through  at 1444 .',
             'pa and lateral views of the chest  at 0923 .','portable semi-upright chest radiograph  0555 .',
             'prior chest radiographs since  most recently  .', 'ap and lateral chest radiographs were obtained .',
             'chest radiograph obtained approximately 3 hr prior .', ' ct torso .']
for pattern in patterns:
    df_filtered = df[df['report'] == pattern]
    print(pattern, df_filtered.shape[0], sep='\n')
filtered_df = df[~df['report'].isin(patterns)]
re_patterns = [r'((is|were|are) submitted (\.|dated))|(is submitted for interpretation \.)', r'chest (film|radiograph)  at ']
for i, re_pattern in enumerate(re_patterns):
    filtered = df[df['report'].str.contains(re_pattern, na=False, regex=True)]
    filtered.sort_values(by='report_length').drop_duplicates(subset='report')['report'].to_csv(f'regex_{i}.csv')
    print(re_pattern, filtered.shape[0], sep='\n')
combined_pattern = f"({'|'.join(re_patterns)})"
filtered2_df = filtered_df[~filtered_df['report'].str.contains(combined_pattern, na=False, regex=True)]
# Sort the DataFrame by the report length
df_sorted = filtered2_df.sort_values(by='report_length')

# Get the top N shortest reports
N = 2000
top_shortest = df_sorted.drop_duplicates(subset='report').head(N)
counts = df_sorted['report'].value_counts()
count_size = counts.shape[0]
count_count = counts.value_counts()
top_shortest[['report_length', 'report']].to_csv(f'findings_{split}.csv')
print(df_sorted.drop_duplicates(subset='report').shape[0])
#print(count_size, top_shortest[['report_length', 'report']], sep='\n')

 .
0
is .
0
am .
0
be .
0
in .
0
at the .
0
see below .
0
send the .
0
see below .
0
 through  .
0
is and for .
0
ct chest of  .
0
chest radiograph  .
0
no previous images .
0
see impression below .
0
prior chest radiographs  .
0
portable chest radiograph .
0
ct scan of the torso from  .
0
radiographs are under penetrated .
0
portable ap upright chest  at 1722 .
0
prior chest radiographs  through 47 .
0
portable ap approach vessel  at 0739 .
0
chest radiographs since  most recently  .
0
prior chest radiographs  through  at 1444 .
0
pa and lateral views of the chest  at 0923 .
0
portable semi-upright chest radiograph  0555 .
0
prior chest radiographs since  most recently  .
0
ap and lateral chest radiographs were obtained .
0
chest radiograph obtained approximately 3 hr prior .
0
 ct torso .
0
((is|were|are) submitted (\.|dated))|(is submitted for interpretation \.)
6
chest (film|radiograph)  at 
2


/tmp/ipykernel_4016262/1311886693.py:29: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df['report'].str.contains(re_pattern, na=False, regex=True)]
/tmp/ipykernel_4016262/1311886693.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered2_df = filtered_df[~filtered_df['report'].str.contains(combined_pattern, na=False, regex=True)]


2148


In [16]:
# Combine test and validation for impressions and findings
if task != 'qa' and split == 'valid':
    df_test = pd.read_csv(f'{ws}/mimic_data/mimic_test_{task}.csv')
    df = pd.concat([df_r, df_test], ignore_index = True)
    print(f'Combined validation and test dataset for task: {task}.\nPrevious size: \
          {df_r.shape[0]}\nNew size: {df.shape[0]}\n')
else:
    df = df_r

In [12]:

# Filter anything that is not antero posterior or postero anterior view
# Step 1: Filter df1 to get dicom_ids where ViewPosition is PA or AP
dicom_ids = df_metadata[df_metadata['ViewPosition'].isin(['PA', 'AP'])]['dicom_id']
# Step 2: Use those dicom_ids to filter df2
filtered_df = df[df['dicom_id'].isin(dicom_ids)]
print(f'Keep only AP and PA views for task: {task}.\nPrevious size: \
          {df.shape[0]}\nNew size: {filtered_df.shape[0]}\n')

Keep only AP and PA views for task: qa.
Previous size:           560563
New size: 560549



In [13]:
# Filter test data from pair question
if task != 'qa':
    test_ids = df_qa[df_qa['split']=='test']

    dicom_id_test = pd.merge(test_ids[['study_id']], df_mimic_all[['study_id', 'dicom_id']], on='study_id', how='inner')['dicom_id']
    clean_df = filtered_df[~filtered_df['dicom_id'].isin(dicom_id_test)]

    print(f'Remove vqa test data for: {task}.\nPrevious size: \
              {filtered_df.shape[0]}\nNew size: {clean_df.shape[0]}\n')
else:
    clean_df = filtered_df

In [ ]:

contents = []

for num_id in tqdm(range(10), desc=f'Processing task {task} and split {split}', unit='images', leave=True):#len(clean_df)):
    row = clean_df.iloc[num_id]
    
    #current image    
    dicom_id = row['dicom_id']
    study_id = str(row['study_id'])
    patient_id = str(row['subject_id'])    
    
    image_path = f'p{patient_id[:2]}/p{patient_id}/s{study_id}/{dicom_id}.jpg'
    image_path = os.path.join(images_path, image_path)
    img = Image.open(image_path) # path to file
    img_buffer = BytesIO()
    img.save(img_buffer, format=img.format)
    byte_data = img_buffer.getvalue()
    base64_str = base64.b64encode(byte_data) # bytes
    base64_str = base64_str.decode("utf-8") # str

    #ref image
    if task == 'qa':
        ref_dicom_id = row['dicom_id_ref']
        ref_study_id = str(int(row['study_id_ref']))
        ref_patient_id = patient_id  
    
        ref_image_path = f'p{ref_patient_id[:2]}/p{ref_patient_id}/s{ref_study_id}/{ref_dicom_id}.jpg'
        ref_image_path = os.path.join(images_path, ref_image_path)
        ref_img = Image.open(ref_image_path) # path to file
        ref_img_buffer = BytesIO()
        ref_img.save(ref_img_buffer, format=ref_img.format)
        ref_byte_data = ref_img_buffer.getvalue()
        ref_base64_str = base64.b64encode(ref_byte_data) # bytes
        ref_base64_str = ref_base64_str.decode("utf-8") # str
        
        question = row['question']
        report = row['answer']
    else:
        ref_base64_str = ''
        report = row['report'] 

    contents.append([str(num_id), base64_str, report, question, ref_base64_str,'', dataset_name,'caption'])

random.shuffle(contents)
f = open(f'{data_dir}/mimic_{task}_{split}_draft.tsv','w', newline='')
wr = csv.writer(f, delimiter='\t')
for content in contents:
    wr.writerow(content)
f.close()


Loop train:



Processing task qa and split train: 100%|██████████| 10/10 [00:03<00:00,  2.82images/s]
